# 16_feature_selection_rfecv_svr_kneedle.ipynb

**Repository:** pulmonary_nodule_verification v1.0.0 · **Date:** 2026-10-26 · **Developer:** Prof. Wesley Pacheco Calixto · **License:** MIT · **Corresponding author:** Wesley Pacheco Calixto, wesley.compute-node

**Citation:** Calixto, W. P., Camargo, T. F. O., Pacheco, V. M. G., Martins, M. S. R., & Coimbra, A. P. (2026). Multiscale morphology-informed post-detection verification of pulmonary nodule candidates in CT: code, trained nnU-Net models, cohort splits and analysis data (v1.0.0) [Data set and software]. Zenodo. https://doi.org/10.5281/zenodo.23674673

Outputs have been cleared; machine-specific paths were neutralised. See docs/file_level_documentation.md.


# 16 — Feature Selection RFECV-SVR + Kneedle

Executa RFECV-SVR linear por output (treino apenas) e aplica cotovelo Kneedle  
para seleccionar as 48 features MG + 12 DG = **60 features totais** do artigo.

### Antes de correr — fazer upload para `Meu Drive > Harmonica > script_16`:
- `15_original_features.parquet`  (10 001 × 177)
- `13_Y_R1_corrigido.parquet`    (10 001 × 22)

### Checkpoint automático
O notebook guarda o progresso após cada output em `script_16/checkpoints/`.  
Se o Colab desconectar, basta correr novamente — retoma do output onde parou.

### Saídas (criadas automaticamente dentro de `script_16/`):
```
script_16/checkpoints/rfecv_checkpoint.pkl   ← progresso por output
script_16/audits/16_selected_features_by_output.csv
script_16/audits/16_feature_ranking.csv
script_16/audits/16_rfecv_scores.csv
script_16/audits/16_kneedle_selection.csv
script_16/figdata/16_feature_ranking_long.csv
script_16/logs/16_feature_selection_rfecv_svr_kneedle.log
script_16/manifests/16_feature_selection_rfecv_svr_kneedle_params.json
```

**Runtime estimado**: 3–6 h (CPU Colab, 22 outputs × 5-fold CV × 177 features)  
**Runtime type**: CPU (SVR é CPU-bound; GPU não ajuda)

In [ ]:
import sklearn
print(f"sklearn {sklearn.__version__}")
assert tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1, 0), \
    "Actualizar: !pip install -U scikit-learn"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

SCRIPT16_DIR = Path("/content/drive/MyDrive/Harmonica/script_16")

IN_FEATURES = SCRIPT16_DIR / "15_original_features.parquet"
IN_Y        = SCRIPT16_DIR / "13_Y_R1_corrigido.parquet"

AUDITS_DIR       = SCRIPT16_DIR / "audits"
FIGDATA_DIR      = SCRIPT16_DIR / "figdata"
LOGS_DIR         = SCRIPT16_DIR / "logs"
MANIFESTS_DIR    = SCRIPT16_DIR / "manifests"
CHECKPOINTS_DIR  = SCRIPT16_DIR / "checkpoints"
CHECKPOINT_FILE  = CHECKPOINTS_DIR / "rfecv_checkpoint.pkl"

for d in [AUDITS_DIR, FIGDATA_DIR, LOGS_DIR, MANIFESTS_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for f in [IN_FEATURES, IN_Y]:
    ok = f.exists()
    print(f"{'✓' if ok else '✗  ← FAZER UPLOAD!'} {f.name}")
    if not ok:
        raise FileNotFoundError(f"Ficheiro não encontrado: {f}")

print(f"\nPasta de saída : {SCRIPT16_DIR}")
print(f"Checkpoint file: {CHECKPOINT_FILE}")

In [ ]:
import json
import logging
import pickle
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(
            LOGS_DIR / "16_feature_selection_rfecv_svr_kneedle.log",
            encoding="utf-8"
        ),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("script16")
print("Imports OK")

In [ ]:
CV_FOLDS    = 5
STEP        = 5
SCORING     = "neg_mean_absolute_error"
N_MG        = 177
N_ADOPTED   = 48
N_DG        = 12
TRAIN_RATIO = 0.80

DG_FEATURE_NAMES = [f"dg_unit_{i:02d}" for i in range(1, N_DG + 1)]

print(f"CV_FOLDS={CV_FOLDS}  STEP={STEP}  N_MG={N_MG}  N_ADOPTED={N_ADOPTED}  N_DG={N_DG}")

In [ ]:
log.info("[1] A carregar X e Y ...")
X_df = pd.read_parquet(IN_FEATURES)
Y_df = pd.read_parquet(IN_Y)

common_idx    = X_df.index.intersection(Y_df.index)
X_df          = X_df.loc[common_idx]
Y_df          = Y_df.loc[common_idx]
feature_names = list(X_df.columns)
output_names  = list(Y_df.columns)
n_samples     = len(X_df)

assert len(feature_names) == N_MG, \
    f"Esperadas {N_MG} features, encontradas {len(feature_names)}"

log.info(f"  X: {X_df.shape}  Y: {Y_df.shape}")
print(f"Outputs ({len(output_names)}): {output_names}")

In [ ]:
train_end      = int(n_samples * TRAIN_RATIO)
X_train        = X_df.iloc[:train_end].values.astype(np.float32)
Y_train        = Y_df.iloc[:train_end].values.astype(np.float32)
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

log.info(f"[2] Treino: {train_end}  Teste: {n_samples - train_end} (não usado no RFECV)")
print(f"X_train_scaled: {X_train_scaled.shape}")

In [ ]:

def reconstruct_n_features_steps(n_features_in, step, min_features, n_scores):
    steps = list(range(min_features, n_features_in + 1, step))
    if not steps or steps[-1] != n_features_in:
        steps.append(n_features_in)
    if len(steps) != n_scores:
        steps = list(map(int, np.round(
            np.linspace(min_features, n_features_in, n_scores)
        )))
    return np.array(steps[:n_scores], dtype=int)

def save_checkpoint(path, last_idx, selected_flags, importances, scores_by_output):
    ckpt = {
        "last_idx":        last_idx,
        "selected_flags":  selected_flags.copy(),
        "importances":     importances.copy(),
        "scores_by_output": [
            (label, nf.tolist(), sc.tolist())
            for label, nf, sc in scores_by_output
        ],
    }
    with open(path, "wb") as f:
        pickle.dump(ckpt, f)

def load_checkpoint(path, n_outputs, n_mg):
    """Devolve (start_i, selected_flags, importances, scores_by_output)."""
    with open(path, "rb") as f:
        ckpt = pickle.load(f)
    selected_flags  = ckpt["selected_flags"]
    importances     = ckpt["importances"]
    scores_by_output = [
        (label, np.array(nf, dtype=int), np.array(sc))
        for label, nf, sc in ckpt["scores_by_output"]
    ]
    return ckpt["last_idx"] + 1, selected_flags, importances, scores_by_output

n_outputs = Y_train.shape[1]

if CHECKPOINT_FILE.exists():
    start_i, selected_flags, importances, scores_by_output = \
        load_checkpoint(CHECKPOINT_FILE, n_outputs, N_MG)
    log.info(f"Checkpoint carregado — retomando do output [{start_i}/{n_outputs}] "
             f"({output_names[start_i] if start_i < n_outputs else 'CONCLUÍDO'})")
else:
    start_i          = 0
    selected_flags   = np.zeros((n_outputs, N_MG), dtype=bool)
    importances      = np.zeros((n_outputs, N_MG), dtype=float)
    scores_by_output = []
    log.info("Sem checkpoint — a iniciar do output 0")

t0 = time.time()
for i in range(start_i, n_outputs):
    label = output_names[i]
    y_i   = Y_train[:, i]
    t_i   = time.time()

    rfecv = RFECV(
        estimator=SVR(kernel="linear", max_iter=5000),
        step=STEP,
        cv=CV_FOLDS,
        scoring=SCORING,
        min_features_to_select=1,
        n_jobs=-1,
    )
    rfecv.fit(X_train_scaled, y_i)

    selected_flags[i] = rfecv.support_
    full_imp = np.zeros(N_MG, dtype=float)
    full_imp[rfecv.support_] = np.abs(rfecv.estimator_.coef_[0])
    importances[i] = full_imp

    cv_scores = rfecv.cv_results_["mean_test_score"]
    nf_steps  = reconstruct_n_features_steps(
        rfecv.n_features_in_, STEP, 1, len(cv_scores)
    )
    scores_by_output.append((label, nf_steps, cv_scores))

    n_sel         = int(rfecv.support_.sum())
    elapsed       = time.time() - t_i
    elapsed_total = time.time() - t0
    n_done        = i - start_i + 1
    n_left        = n_outputs - i - 1
    eta           = elapsed_total / n_done * n_left if n_done > 1 else 0
    log.info(f"  [{i+1:2d}/{n_outputs}] {label}: {n_sel} features  "
             f"({elapsed:.0f}s | ETA {eta/60:.0f}min)")

    save_checkpoint(CHECKPOINT_FILE, i, selected_flags, importances, scores_by_output)

log.info(f"RFECV concluído em {(time.time()-t0)/60:.1f} min (esta sessão)")

In [ ]:
union_mask = selected_flags.any(axis=0)
n_union    = int(union_mask.sum())

selected_feature_names = [feature_names[j] for j in range(N_MG) if union_mask[j]]
imp_union  = importances[:, union_mask]
freq_union = selected_flags[:, union_mask].sum(axis=0)

log.info(f"[4] Union: {n_union} features de {N_MG}")

In [ ]:
mean_imp_union = imp_union.mean(axis=0)
sort_idx       = np.argsort(mean_imp_union)[::-1]

features_sorted = [selected_feature_names[j] for j in sort_idx]
imp_values      = mean_imp_union[sort_idx]
freq_sorted     = freq_union[sort_idx]

cumsum      = np.cumsum(imp_values)
cumsum_norm = cumsum / cumsum[-1]
x_norm      = np.linspace(0, 1, n_union)
distances   = cumsum_norm - x_norm
knee_idx    = int(np.argmax(distances))
n_knee      = knee_idx + 1
pct_knee    = float(cumsum_norm[knee_idx] * 100)
pct_adopted = float(cumsum_norm[N_ADOPTED - 1] * 100)

log.info(f"[5] Joelho Kneedle: feature #{n_knee}  ({pct_knee:.1f}%)")
log.info(f"    Threshold adotado: feature #{N_ADOPTED}  ({pct_adopted:.1f}%)")
log.info(f"    Feature no joelho: '{features_sorted[knee_idx]}'")

In [ ]:
features_adopted   = features_sorted[:N_ADOPTED]
all_model_features = features_adopted + DG_FEATURE_NAMES
n_model            = len(all_model_features)

if n_model == 60:
    log.info(f"[6] Total X: {n_model}  ✓ Reprodução 177 → 60 confirmada")
else:
    log.warning(f"[6] Total X: {n_model}  ⚠ artigo reporta 60")

In [ ]:

sel_rows = []
for i, label in enumerate(output_names):
    for j in range(N_MG):
        if selected_flags[i, j]:
            sel_rows.append({"output": label, "feature": feature_names[j]})
pd.DataFrame(sel_rows).to_csv(
    AUDITS_DIR / "16_selected_features_by_output.csv", index=False
)
log.info(f"✓ 16_selected_features_by_output.csv  ({len(sel_rows)} linhas)")

rank_rows = []
for rank, (feat, imp, freq) in enumerate(
    zip(features_sorted, imp_values, freq_sorted), start=1
):
    rank_rows.append({
        "rank":         rank,
        "feature":      feat,
        "mean_imp":     round(float(imp), 6),
        "frequency":    int(freq),
        "pct_cumul":    round(float(cumsum_norm[rank-1] * 100), 2),
        "kneedle_dist": round(float(distances[rank-1]), 6),
        "in_adopted":   rank <= N_ADOPTED,
    })
pd.DataFrame(rank_rows).to_csv(AUDITS_DIR / "16_feature_ranking.csv", index=False)
log.info(f"✓ 16_feature_ranking.csv  ({len(rank_rows)} linhas)")

score_rows = []
for label, nf_steps, cv_scores in scores_by_output:
    for nf, sc in zip(nf_steps, cv_scores):
        score_rows.append({
            "output": label, "n_features": int(nf), "mae_cv": round(float(-sc), 6)
        })
pd.DataFrame(score_rows).to_csv(AUDITS_DIR / "16_rfecv_scores.csv", index=False)
log.info(f"✓ 16_rfecv_scores.csv  ({len(score_rows)} linhas)")

pd.DataFrame([{
    "n_union": n_union, "knee_n": n_knee, "knee_pct": pct_knee,
    "knee_feature": features_sorted[knee_idx], "adopted_n": N_ADOPTED,
    "adopted_pct": pct_adopted, "n_dg": N_DG, "n_model": n_model,
}]).to_csv(AUDITS_DIR / "16_kneedle_selection.csv", index=False)
log.info("✓ 16_kneedle_selection.csv")

In [ ]:
fig_rows = []

for label, nf_steps, cv_scores in scores_by_output:
    for nf, sc in zip(nf_steps, cv_scores):
        fig_rows.append({
            "record_type": "mae_curve", "output": label,
            "n_features": int(nf), "mae_cv": round(float(-sc), 6),
            "feature": None, "frequency": None, "coef_abs": None,
        })

feature_to_freq = {
    features_sorted[k]: int(freq_sorted[k]) for k in range(len(features_sorted))
}
for i, label in enumerate(output_names):
    for feat in features_sorted:
        fi = feature_names.index(feat)
        fig_rows.append({
            "record_type": "feature_importance", "output": label,
            "n_features": None, "mae_cv": None,
            "feature": feat, "frequency": feature_to_freq[feat],
            "coef_abs": round(float(importances[i, fi]), 8),
        })

fig_df = pd.DataFrame(fig_rows)
fig_df.to_csv(FIGDATA_DIR / "16_feature_ranking_long.csv", index=False)
log.info(f"✓ figdata/16_feature_ranking_long.csv  shape={fig_df.shape}")

In [ ]:
manifest = {
    "script":        "16_feature_selection_rfecv_svr_kneedle.py",
    "run_timestamp": datetime.now().isoformat(),
    "parameters": {
        "cv_folds": CV_FOLDS, "step": STEP, "scoring": SCORING,
        "n_mg": N_MG, "n_adopted": N_ADOPTED,
        "n_dg": N_DG, "train_ratio": TRAIN_RATIO,
    },
    "results": {
        "n_samples": n_samples, "train_end": train_end,
        "n_outputs": n_outputs, "n_union_features": n_union,
        "knee_n_features": n_knee, "knee_pct": pct_knee,
        "adopted_n": N_ADOPTED, "adopted_pct": pct_adopted,
        "n_model_total": n_model,
        "features_adopted": features_adopted,
        "dg_features": DG_FEATURE_NAMES,
        "all_model_features": all_model_features,
    },
    "inputs": {"features": str(IN_FEATURES), "Y": str(IN_Y)},
    "outputs": {
        "folder": str(SCRIPT16_DIR),
        "audits": [
            "audits/16_selected_features_by_output.csv",
            "audits/16_feature_ranking.csv",
            "audits/16_rfecv_scores.csv",
            "audits/16_kneedle_selection.csv",
        ],
        "figdata": ["figdata/16_feature_ranking_long.csv"],
    },
}
mp = MANIFESTS_DIR / "16_feature_selection_rfecv_svr_kneedle_params.json"
with open(mp, "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, ensure_ascii=False, default=str)

log.info(f"✓ Manifesto: {mp}")
log.info("")
log.info("=== SUMÁRIO ===")
log.info(f"  Entrada:  {N_MG} features MG × {n_outputs} outputs")
log.info(f"  Union:    {n_union} features")
log.info(f"  Joelho:   feature #{n_knee} ({pct_knee:.1f}%)")
log.info(f"  Adotado:  {N_ADOPTED} features ({pct_adopted:.1f}%)")
log.info(f"  DG zeros: {N_DG} features")
log.info(f"  Total X:  {n_model} features")
log.info("=== Concluído ===")

In [ ]:
import pickle, json, pandas as pd
from pathlib import Path

SCRIPT16_DIR    = Path("/content/drive/MyDrive/Harmonica/script_16")
CHECKPOINT_FILE = SCRIPT16_DIR / "checkpoints" / "rfecv_checkpoint.pkl"

if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "rb") as f:
        ckpt = pickle.load(f)
    last_idx = ckpt["last_idx"]
    n_scores = len(ckpt["scores_by_output"])
    print(f"✓ Checkpoint: último output concluído = índice {last_idx}  ({n_scores} de 22)")
else:
    print("✗ Checkpoint não encontrado")

files = [
    "audits/16_kneedle_selection.csv",
    "audits/16_feature_ranking.csv",
    "audits/16_rfecv_scores.csv",
    "audits/16_selected_features_by_output.csv",
    "figdata/16_feature_ranking_long.csv",
    "manifests/16_feature_selection_rfecv_svr_kneedle_params.json",
    "logs/16_feature_selection_rfecv_svr_kneedle.log",
]
print()
for f in files:
    p = SCRIPT16_DIR / f
    print(f"{'✓' if p.exists() else '✗'} {f}")

kneedle_csv = SCRIPT16_DIR / "audits/16_kneedle_selection.csv"
if kneedle_csv.exists():
    print()
    print(pd.read_csv(kneedle_csv).to_string(index=False))

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

SCRIPT16_DIR = Path("/content/drive/MyDrive/Harmonica/script_16")
TMP_DIR      = Path("/content/script16_outputs")

for folder in ["audits", "figdata", "logs", "manifests"]:
    src = SCRIPT16_DIR / folder
    dst = TMP_DIR / folder
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        n = len(list(src.rglob("*")))
        print(f"✓ {folder}/  ({n} ficheiros)")

zip_path = "/content/Harmonic_Project_script16"
shutil.make_archive(zip_path, "zip", TMP_DIR)
print(f"\nZip pronto — a fazer download...")
files.download(zip_path + ".zip")